# Feature tables

| Part | Output | Time (approx.) |
|---|---|---|
| `base` | `work/base_train.parquet`, `work/base_test.parquet`: ~150 columns per departure | 1 min |
| `groups` | `work/groups/*.parquet`: queues, delay indices, arrivals / METAR, forward windows, stand arrivals | 1–2 min |
| `adsb` | `work/adsb_tracks/<day>.parquet`: ADS-B track features (6 months of the training year + test months) | 3 min |
| `adsb_sparse` | `work/adsb_tracks_s30`, `adsb_tracks_s10`: thinned copies (only needed to retrain the ADS-B corrector) | 5 min |
| `opdi` | `work/groups/opdi.parquet`: OPDI ground-event features | 1 min |

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
import config
import run
print("raw data folder:", config.RAW.name)

raw data folder: data


In [2]:
PARTS = ["base", "groups", "adsb", "opdi"]
REBUILD = False

In [3]:
from taxi.features import tables as T
present = {"base": (config.WORK / "base_test.parquet").exists(), "groups": (T.GROUP_DIR / "queue.parquet").exists(),
           "adsb": (config.WORK / "adsb_tracks").exists(), "adsb_sparse": (config.WORK / "adsb_tracks_s10").exists(),
           "opdi": (T.GROUP_DIR / "opdi.parquet").exists()}
todo = [p for p in PARTS if REBUILD or not present[p]]
print("to build:", todo or "nothing (all present)")
if todo:
    run.features(*todo)

to build: nothing (all present)


## Checks

In [4]:
e, t = T.base("train"), T.base("test")
print("train", e.shape, "| test", t.shape)
cols = ["NM_TAXI_DIFF_SEC", "DEP_QUEUE_N", "TEMPERATURE", "BASE_TAXI_TIME", "STAND_TO_RWY_DIST_M", "RWY_PHYSICAL_LENGTH_FT"]
pd.DataFrame({"train missing share": e[cols].isna().mean(), "test missing share": t[cols].isna().mean()}).round(4)

train (2085047, 156) | test (670790, 156)


,train missing share,test missing share
NM_TAXI_DIFF_SEC,0.0108,0.0135
DEP_QUEUE_N,0.0000,0.0000
TEMPERATURE,0.0000,0.0000
BASE_TAXI_TIME,0.0000,0.0000
STAND_TO_RWY_DIST_M,0.0000,0.0000
RWY_PHYSICAL_LENGTH_FT,0.0002,0.0002


In [5]:
t.groupby("MVT_MONTH")[["NM_TAXI_DIFF_SEC", "DEP_QUEUE_N", "TEMPERATURE", "Q_ARR_30"]].mean().round(1)

,NM_TAXI_DIFF_SEC,DEP_QUEUE_N,TEMPERATURE,Q_ARR_30
MVT_MONTH,,,,
1,1077.7,9.7,5.000000,13.4
2,1013.2,9.6,8.300000,13.9
6,992.5,10.1,23.200001,15.8
7,1003.7,10.3,25.100000,16.1


In [6]:
from taxi.correctors import common as C
tracks = C.adsb_features("adsb_tracks", columns=["MVT_ID_mvt", "ADSB_MATCH"])
opdi = T.table("opdi")[["MVT_ID_mvt"]].assign(opdi=1)
k = pd.concat([e[["MVT_ID_mvt", "MVT_MONTH"]].assign(year=2025), t[["MVT_ID_mvt", "MVT_MONTH"]].assign(year=2026)])
k = k.merge(tracks, on="MVT_ID_mvt", how="left").merge(opdi.astype({"MVT_ID_mvt": "float64"}), on="MVT_ID_mvt", how="left")
k[k.ADSB_MATCH.notna()].groupby(["year", "MVT_MONTH"]).agg(
    departures=("MVT_ID_mvt", "size"), adsb_matched=("ADSB_MATCH", lambda s: (s > 0).mean()),
    opdi_matched=("opdi", lambda s: s.notna().mean())).round(3)

departures  adsb_matched  opdi_matched
year MVT_MONTH                                        
2025 1              153706         0.607         0.548
     2              143732         0.593         0.616
     6              183142         0.739         0.528
     7              190713         0.734         0.488
     8              191182         0.728         0.473
     12             165677         0.765         0.527
2026 1              152719         0.724         0.328
     2              144158         0.715         0.501
     6              181791         0.696         0.594
     7              192122         0.707         0.590